# Lectura de paquetes y data

In [ ]:
import warnings
import os
import time
import sys
import ast

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from statsmodels.tsa.seasonal import STL

from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNet
from sklearn.neighbors import KNeighborsRegressor

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import mean_absolute_error, mean_squared_error

import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)  # Reduce verbosity de Optuna


# Agrega todo el directorio padre al path
sys.path.append(os.path.abspath(".."))
from src.utils_ml import ml_training_utils as ml_utils
from src.utils_ml import ml_feature_engineering as fe_utils
from src.utils_ml import ml_plotting as plot_utils

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
warnings.filterwarnings("ignore")

Utilizamos paths relativos para la lectura de la data

In [ ]:
filename = "ml_pipeline_p_sku.ipynb"  # nombre del archivo actual
print(f"Current absolute path: {os.getcwd()}\n")

# Especificamos la ruta del directorio actual y los directorios de datos y salida
ACTUAL_DIR = os.path.dirname(os.path.abspath(filename))
BASE_DIR = os.path.dirname(ACTUAL_DIR)
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUT_DIR = os.path.join(DATA_DIR, "output")

print(f"BASE_DIR: {BASE_DIR}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

In [ ]:
# Cargar el archivo de Excel
file_path = os.path.join(DATA_DIR, "data_demanda.xlsx")
df_base = pd.read_excel(file_path, sheet_name="data")
df_base = df_base.drop("Cliente", axis=1)

df_base.shape

In [ ]:
df_base.head(5)

In [ ]:
# Filtrar los datos relevantes para este analisis

df = (
    df_base[["Fe.prefer.entrega", "SKU", "Pedidos"]]
    .copy()
    .rename(
        columns={
            "Fe.prefer.entrega": "Fecha",
        }
    )
)
df["Pedidos"] = pd.to_numeric(df["Pedidos"], errors="coerce")

In [ ]:
df

# Preparación de la data

In [ ]:
### Primero, nos aseguramos de que se cuente un dato por SKU por dia
# -------

# rango completo de fechas desde la más antigua hasta la más reciente
fecha_min = df["Fecha"].min()
fecha_max = df["Fecha"].max()
rango_fechas = pd.date_range(start=fecha_min, end=fecha_max, freq="D")

# Obtenemos todos los SKUs únicos
skus = df["SKU"].unique()

# DataFrame con todas las combinaciones de SKU y fecha
combinaciones_completas = pd.MultiIndex.from_product(
    [rango_fechas, skus], names=["Fecha", "SKU"]
).to_frame(index=False)

# Unir con el dataframe original para rellenar con ceros donde falten datos
df_completo = combinaciones_completas.merge(df, on=["Fecha", "SKU"], how="left")

# Rellenar valores faltantes de pedidos con 0
df_completo["Pedidos"] = df_completo["Pedidos"].fillna(0).astype(int)

# Ordenar por SKU y Fecha (opcional)
df_completo = df_completo.sort_values(["SKU", "Fecha"]).reset_index(drop=True)

df = df_completo.copy()

In [ ]:
# Modificar nombre de columnas
df.columns = df.columns.str.replace(".", "_", regex=False).str.lower()

In [ ]:
df.shape

# EDA

## general

In [ ]:
df.isna().sum()

In [ ]:
# porcentaje de ceros por sku
porcentaje_ceros = (
    df.groupby("sku")["pedidos"]
    .apply(lambda x: (x == 0).mean() * 100)
    .reset_index(name="prct_ceros")
    .round(2)
)

# promedio, mediana y desviacion estandar por sku excluyendo ceros
df_temp = df[df["pedidos"] > 0].copy()
promedio = df_temp.groupby("sku")["pedidos"].mean().reset_index(name="Promedio").round()
mediana = df_temp.groupby("sku")["pedidos"].median().reset_index(name="Mediana").round()
desviacion = (
    df_temp.groupby("sku")["pedidos"].std().reset_index(name="Desviacion").round()
)
maximo = df_temp.groupby("sku")["pedidos"].max().reset_index(name="Maximo").round()

# Porcentaje de valores outliers por SKU excluyendo ceros
porcentaje_outliers = (
    df_temp.groupby("sku")["pedidos"]
    .apply(fe_utils.calcular_outliers_porcentaje)
    .reset_index(name="prct_outliers")
    .round(2)
)

# Unir las tablas
tabla_total = pd.merge(porcentaje_ceros, porcentaje_outliers, on="sku", how="outer")
tabla_total = pd.merge(tabla_total, promedio, on="sku", how="outer")
tabla_total = pd.merge(tabla_total, mediana, on="sku", how="outer")
tabla_total = pd.merge(tabla_total, desviacion, on="sku", how="outer")
tabla_total = pd.merge(tabla_total, maximo, on="sku", how="outer")
tabla_total.sort_values(by="prct_ceros", ascending=False)

## Tendencias

In [ ]:
sku = "SKU5"
print(f"Analizando el SKU: {sku}")

In [ ]:
df_sku = df[df["sku"] == sku].copy()
df_sku = df_sku.drop("sku", axis=1)

# graficamos la serie de tiempo del SKU seleccionado usando plotly
fig = px.line(
    df_sku,
    x="fecha",
    y="pedidos",
    title=f"Serie de tiempo de Pedidos para {sku}:",
)
fig.update_layout(
    xaxis_title="Fecha",
)
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor="LightGray")
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor="LightGray")
fig.update_traces(line=dict(color="blue", width=2))
fig.show()


### Descomposición STL

In [ ]:
# graficamos la tendencia y estacionalidad de cada SKU usando STL
plot_utils.graficar_serie_con_descomposicion(df_sku, sku=sku, periodo=7)


# Feature engineering

## Variables temporales

In [ ]:
df = fe_utils.create_temporal_features(df, "fecha")
df.shape, df.columns

## Variables tipo lag

In [ ]:
df = fe_utils.create_lag_features(
    df, "pedidos", "sku", "fecha", max_daily_lag=14, weekday_lags=3
)
df.shape, df.columns

## Variables tipo promedio moviles

In [ ]:
df = fe_utils.create_rolling_features(df, "pedidos", "sku", "fecha")
df.shape, df.columns

## Variables lags de STL

In [ ]:
df = fe_utils.create_stl_features(
    df, "pedidos", "sku", "fecha", seasonal=7, stl_lags=14
)
df.shape, df.columns


## Aplanamiento de outliers en demanda

In [ ]:
df = fe_utils.cap_upper_outliers(df, "pedidos", "sku")

## Ajustes finales a la data

In [ ]:
# Eliminamos todas las filas con valores NaN para que no afecten el entrenamiento
df.dropna(inplace=True)
df.shape, df.columns

In [ ]:
df

# Modelling 

In [ ]:
## Preparación de data para modelling

# Identificamos las variables predictoras
features = df.columns.difference(["fecha", "sku", "pedidos"]).to_list()

# definimos el numero de ventanas de evaluación y el tamaño de las ventanas
val_iter = 3
val_size = 7

## Evaluación modelos XGBoost

In [ ]:
# Creamos un DataFrame para almacenar los resultados de cada SKU
# Este DataFrame contendrá el SKU, los parámetros del modelo y las métricas de evaluación
##########

results = []

for sku in df["sku"].unique():
    print("\n-----------------------")
    print(f"\n🔍 Optimizando para SKU: {sku}\n")

    # Filtramos el DataFrame para el SKU actual
    df_sku = df[df["sku"] == sku].copy()

    # eliminamos los ultimos x dias, para una ultima iteracion out of sample para test
    df_sku = df_sku[:-val_size]

    # Calculamos el tamaño del conjunto de entrenamiento
    train_size = df_sku.shape[0] - val_iter * val_size

    # Definimos el espacio de búsqueda de hiperparámetros para el modelo XGBoost
    # Usamos funciones lambda para que Optuna pueda sugerir valores
    param_grid = {
        "max_depth": lambda trial: trial.suggest_int("max_depth", 1, 5, step=1),
        "learning_rate": lambda trial: trial.suggest_float(
            "learning_rate", 0.001, 0.1, step=0.001
        ),
        "n_estimators": lambda trial: trial.suggest_int(
            "n_estimators", 50, 3000, step=50
        ),
        "subsample": lambda trial: trial.suggest_float(
            "subsample", 0.5, 1.0, step=0.02
        ),
        "colsample_bytree": lambda trial: trial.suggest_float(
            "colsample_bytree", 0.5, 1.0, step=0.02
        ),
        "gamma": lambda trial: trial.suggest_float("gamma", 1, 30, step=0.5),
        "reg_alpha": lambda trial: trial.suggest_float("reg_alpha", 1, 30, step=0.5),
        "reg_lambda": lambda trial: trial.suggest_float("reg_lambda", 1, 30, step=0.5),
        "min_child_weight": lambda trial: trial.suggest_int(
            "min_child_weight", 5, 20, step=1
        ),
        "random_state": lambda trial: 100,  # Fijamos la semilla para reproducibilidad
    }

    # Optimizamos el modelo XGBoost usando Optuna con ventana recursiva
    study = ml_utils.optimize_model_with_optuna(
        model_class=XGBRegressor,
        param_grid=param_grid,
        X=df_sku[features],
        y=df_sku["pedidos"],
        n_trials=100,
        val_iter=val_iter,
        train_size=train_size,
        val_size=val_size,
        show_progress_bar=True,
    )

    # Obtenemos el mejor trial del estudio
    # y almacenamos los resultados en un diccionario
    best_trial = study.best_trials[0]
    results.append(
        {
            "sku": sku,
            "study": study,
            "best_params": best_trial.params,
            "best_smape": best_trial.values[0],
            "best_gap": best_trial.values[1],
            "model": "XGBRegressor",
            "n_trials": len(study.trials),
        }
    )

    print("\n-----------------------")


df_results_xgboost = pd.DataFrame(results)

In [ ]:
df_results_xgboost

In [ ]:
## Mostrar graficas de Optuna por estudio especifico

# sku = "SKU1"
# temp = df_results_xgboost[df_results_xgboost["sku"] == sku].copy()
# plot_utils.mostrar_graficas_optuna(temp["study"], temp["model"])

## Evaluación modelos Random Forest

In [ ]:
# Creamos un DataFrame para almacenar los resultados de cada SKU
# Este DataFrame contendrá el SKU, los parámetros del modelo y las métricas de evaluación
##########

results = []


for sku in df["sku"].unique():
    print("\n-----------------------")
    print(f"\n🔍 Optimizando para SKU: {sku}\n")

    # Filtramos el DataFrame para el SKU actual
    df_sku = df[df["sku"] == sku].copy()

    # eliminamos los ultimos x dias, para una ultima iteracion out of sample para test
    df_sku = df_sku[:-val_size]

    # Calculamos el tamaño del conjunto de entrenamiento
    train_size = df_sku.shape[0] - val_iter * val_size

    # Espacio de búsqueda de hiperparámetros para Random Forest
    param_grid = {
        "n_estimators": lambda trial: trial.suggest_int(
            "n_estimators", 50, 1000, step=50
        ),
        "max_depth": lambda trial: trial.suggest_int("max_depth", 1, 10, step=1),
        "min_samples_split": lambda trial: trial.suggest_int(
            "min_samples_split", 2, 10
        ),
        "min_samples_leaf": lambda trial: trial.suggest_int(
            "min_samples_leaf", 3, 15, step=1
        ),
        "max_features": lambda trial: trial.suggest_categorical(
            "max_features", ["sqrt", "log2", None]
        ),
        "bootstrap": lambda trial: trial.suggest_categorical(
            "bootstrap", [True, False]
        ),
        "random_state": lambda trial: 100,  # Fijamos el random_state para reproducibilidad
    }

    # Optimizamos usando tu función personalizada con modelo RandomForestRegressor
    study = ml_utils.optimize_model_with_optuna(
        model_class=RandomForestRegressor,
        param_grid=param_grid,
        X=df_sku[features],
        y=df_sku["pedidos"],
        n_trials=100,
        val_iter=val_iter,
        train_size=train_size,
        val_size=val_size,
        show_progress_bar=True,
    )

    # Registramos los resultados
    best_trial = study.best_trials[0]
    results.append(
        {
            "sku": sku,
            "study": study,
            "best_params": best_trial.params,
            "best_smape": best_trial.values[0],
            "best_gap": best_trial.values[1],
            "model": "RandomForestRegressor",
            "n_trials": len(study.trials),
        }
    )

    print("-----------------------")

# Creamos DataFrame con resultados
df_results_rf = pd.DataFrame(results)


In [ ]:
df_results_rf

In [ ]:
## Mostrar graficas de Optuna por estudio especifico

# sku = "SKU1"
# temp = df_results_rf[df_results_rf["sku"] == sku].copy()
# plot_utils.mostrar_graficas_optuna(temp["study"], temp["model"])

## Evaluación modelos Elastic Net

In [ ]:
# Creamos un DataFrame para almacenar los resultados de cada SKU
# Este DataFrame contendrá el SKU, los parámetros del modelo y las métricas de evaluación
##########

results = []


for sku in df["sku"].unique():
    print("\n-----------------------")
    print(f"\n🔍 Optimizando para SKU: {sku}\n")

    # Filtramos el DataFrame para el SKU actual
    df_sku = df[df["sku"] == sku].copy()

    # eliminamos los ultimos x dias, para una ultima iteracion out of sample para test
    df_sku = df_sku[:-val_size]

    # Calculamos el tamaño del conjunto de entrenamiento
    train_size = df_sku.shape[0] - val_iter * val_size

    # Espacio de búsqueda de hiperparámetros para ElasticNet
    param_grid = {
        "model__alpha": lambda trial: trial.suggest_float(
            "model__alpha", 0.0001, 1000.0, log=True
        ),
        "model__l1_ratio": lambda trial: trial.suggest_float(
            "model__l1_ratio", 0.0, 1.0, step=0.01
        ),  # 0 = Ridge, 1 = Lasso
        "model__random_state": lambda trial: 100,  # Fijamos el random_state para reproducibilidad
    }

    # Pipeline: escalado + modelo
    model_pipeline = Pipeline(
        [("scaler", StandardScaler()), ("model", ElasticNet(max_iter=20000))]
    )

    # Optimización con tu función
    study = ml_utils.optimize_model_with_optuna(
        model_class=lambda **params: model_pipeline.set_params(**params),
        param_grid=param_grid,
        X=df_sku[features],
        y=df_sku["pedidos"],
        n_trials=200,
        val_iter=val_iter,
        train_size=train_size,
        val_size=val_size,
        show_progress_bar=True,
    )

    best_trial = study.best_trials[0]
    results.append(
        {
            "sku": sku,
            "study": study,
            "best_params": best_trial.params,
            "best_smape": best_trial.values[0],
            "best_gap": best_trial.values[1],
            "model": "ElasticNet",
            "n_trials": len(study.trials),
        }
    )

    print("-----------------------")

df_results_elastic = pd.DataFrame(results)


In [ ]:
df_results_elastic

In [ ]:
## Mostrar graficas de Optuna por estudio especifico

# sku = "SKU1"
# temp = df_results_elastic[df_results_elastic["sku"] == sku].copy()
# plot_utils.mostrar_graficas_optuna(temp["study"], temp["model"])

## Evaluación modelos KNeighbors 

In [ ]:
# Creamos un DataFrame para almacenar los resultados de cada SKU
# Este DataFrame contendrá el SKU, los parámetros del modelo y las métricas de evaluación
##########

results = []


for sku in df["sku"].unique():
    print("\n-----------------------")
    print(f"\n🔍 Optimizando para SKU: {sku}\n")

    # Filtramos el DataFrame para el SKU actual
    df_sku = df[df["sku"] == sku].copy()

    # eliminamos los ultimos x dias, para una ultima iteracion out of sample para test
    df_sku = df_sku[:-val_size]

    # Calculamos el tamaño del conjunto de entrenamiento
    train_size = df_sku.shape[0] - val_iter * val_size

    # Pipeline: escalado obligatorio + modelo
    pipeline = Pipeline(
        [("scaler", StandardScaler()), ("model", KNeighborsRegressor())]
    )

    # Espacio de búsqueda
    param_grid = {
        "model__n_neighbors": lambda trial: trial.suggest_int(
            "model__n_neighbors", 2, 30, step=1
        ),
        "model__weights": lambda trial: trial.suggest_categorical(
            "model__weights", ["uniform", "distance"]
        ),
        "model__p": lambda trial: trial.suggest_int(
            "model__p", 1, 2
        ),  # 1 = manhattan, 2 = euclidean
        "model__leaf_size": lambda trial: trial.suggest_int(
            "model__leaf_size", 5, 100, step=1
        ),
        "model__algorithm": lambda trial: trial.suggest_categorical(
            "model__algorithm", ["auto", "ball_tree", "kd_tree"]
        ),
    }

    # Llamada a tu función personalizada
    study = ml_utils.optimize_model_with_optuna(
        model_class=lambda **params: pipeline.set_params(**params),
        param_grid=param_grid,
        X=df_sku[features],
        y=df_sku["pedidos"],
        n_trials=400,
        val_iter=val_iter,
        train_size=train_size,
        val_size=val_size,
        show_progress_bar=True,
    )

    best_trial = study.best_trials[0]
    results.append(
        {
            "sku": sku,
            "study": study,
            "best_params": best_trial.params,
            "best_smape": best_trial.values[0],
            "best_gap": best_trial.values[1],
            "model": "KNeighborsRegressor",
            "n_trials": len(study.trials),
        }
    )

    print("-----------------------")

df_results_knn = pd.DataFrame(results)


In [ ]:
df_results_knn

In [ ]:
## Mostrar graficas de Optuna por estudio especifico

# sku = "SKU1"
# temp = df_results_knn[df_results_knn["sku"] == sku].copy()
# plot_utils.mostrar_graficas_optuna(temp["study"], temp["model"])

## Selección mejor modelo y ajuste final

In [ ]:
# Eliminamos los estudios de Optuna de todos los DataFrames de resultados
# df_results_xgboost = df_results_xgboost.drop(columns=["study"])
# df_results_rf = df_results_rf.drop(columns=["study"])
# df_results_elastic = df_results_elastic.drop(columns=["study"])
# df_results_knn = df_results_knn.drop(columns=["study"])

# Union de resultados de los modelos
df_results = pd.concat(
    [df_results_xgboost, df_results_rf, df_results_elastic, df_results_knn],
    ignore_index=True,
).sort_values(by=["sku", "best_smape"], ascending=[True, True])

# Por cada SKU, obtenemos el mejor modelo
df_best_models = df_results.loc[
    df_results.groupby("sku")["best_smape"].idxmin()
].reset_index(drop=True)


# Ordenamos por mejor SMAPE
df_best_models = df_best_models.sort_values(by="best_smape", ascending=True).drop(
    columns=["study"]
)

In [ ]:
# guardar en OUTPUT_DIR el dataframe de resultados por SKU
output_file = os.path.join(OUTPUT_DIR, "df_ML_models_results.xlsx")
df_results.to_excel(output_file, index=False)

df_results

In [ ]:
# guardar en OUTPUT_DIR el dataframe de mejores modelos por SKU
output_file = os.path.join(OUTPUT_DIR, "df_ML_best_models.xlsx")
df_best_models.to_excel(output_file, index=False)

df_best_models

# Predicción y graficas 

In [ ]:
# Cargar tabla con mejor modelo por SKU
df_best_models_2 = pd.read_excel(
    os.path.join(OUTPUT_DIR, "df_ML_best_models.xlsx"),
    engine="openpyxl",
)

In [ ]:
df_best_models_2.sample(2)

In [ ]:
## Preparación

# Identificamos las variables predictoras
features = df.columns.difference(["fecha", "sku", "pedidos"]).to_list()

# definimos el tamaño de la ventana de test
test_size = 7

## Tabla de predicciones finales

In [ ]:
## Reentrenamos la serie de tiempo de cada SKU con el mejor modelo,
# usando el conjunto de entrenamiento completo,
# y predecimos sobre el test set para obtener metricas y predicciones
#####

df_test_results = ml_utils.evaluate_models_on_test(
    df=df,
    df_best_models=df_best_models_2,
    target_col="pedidos",
    feature_cols=features,
    sku_col="sku",
    test_size=7,
)

df_test_results = df_test_results.sort_values(by="SKU", ascending=True)

In [ ]:
# Guardar resultado de los modelos sobre el test set
output_file = os.path.join(OUTPUT_DIR, "forecast_results_ml.xlsx")
df_test_results.to_excel(output_file, index=False)

df_test_results

## Grafica de predicciones finales

In [ ]:
plot_utils.plot_forecast_per_sku(
    df_full=df,
    df_test_results=df_test_results,
    target_col="pedidos",
    sku_col="sku",
    test_size=test_size,
    n_train_days=test_size
    * 5,  # Usamos 5 veces el tamaño del test set para el entrenamiento
)
